In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

books = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final.csv", encoding= 'latin-1')

In [9]:
books.head(5)

,ISBN,Title,Author,Year,Publisher,Genre,Description
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,Social Science,Provides an introduction to classical myths pl...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,Actresses,"In a small town in Canada, Clara Callan reluct..."
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,History,"Here, for the first time in paperback, is an o..."
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,Medical,"""Scientists have recently discovered shards of..."
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton & Company,Design,A look at the incredibly well-preserved ancien...


In [23]:
# books['Author'].fillna('Unknown', inplace=True)
books['Publisher'].fillna('Unknown', inplace=True)

/tmp/ipykernel_8605/1324376985.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  books['Publisher'].fillna('Unknown', inplace=True)


In [37]:
books['Genre'] = books['Genre'].fillna("Unknown Genre")
books['Description'] = books['Description'].fillna("No description available.")


In [39]:
books['Description'].isna().sum()

0

In [41]:
books['text_for_embedding'] = books['Title'] + ' ' + books['Author'] + ' ' + books['Publisher'] + ' ' + books['Genre'] + ' ' + books['Description']
books[['ISBN', 'text_for_embedding']].to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final_for_embedding.csv", index=False, encoding='latin-1')

In [42]:
books_f = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final_for_embedding.csv", encoding= 'latin-1')
books_f.head(5)

,ISBN,text_for_embedding
0,0195153448,Classical Mythology Mark P. O. Morford Oxford ...
1,0002005018,Clara Callan Richard Bruce Wright HarperFlamin...
2,0060973129,Decision in Normandy Carlo D'Este HarperPerenn...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...
4,0393045218,The Mummies of Urumchi E. J. W. Barber W. W. N...


In [44]:
books_f.shape

(271379, 2)

In [2]:

import pandas as pd

df = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/books_embeddings.csv", encoding= 'latin-1')

print(type(df["embeddings"].iloc[0]))


<class 'str'>


In [2]:
import numpy as np
import pandas as pd
import ast

df = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/books_embeddings.csv")

# Convert string → list → numpy array safely
embeddings_array = np.array([np.array(ast.literal_eval(e)) for e in df["embeddings"]])

# Save embeddings
np.save("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/embeddings.npy", embeddings_array)

# Save ISBN list
df["ISBN"].to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/isbn_list.csv", index=False)


In [4]:
import pandas as pd

df = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final.csv")  # your main original raw dataset
df["text_for_embedding"] = (
    df["Title"].fillna("") + " " +
    df["Author"].fillna("") + " " +
    df["Genre"].fillna("") + " " +
    df["Description"].fillna("")
)

df.to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_full.csv", index=False)


In [3]:
import pandas as pd

x = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final.csv")
x['ISBN'].duplicated().sum()

0

In [4]:
import pandas as pd

books_meta = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_full.csv")

# Drop duplicates based on Title + Author (keep the first edition)
books_meta = books_meta.drop_duplicates(subset=["Title", "Author"], keep="first")

books_meta.to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv", index=False)

print("✅ Cleaned metadata saved!")


✅ Cleaned metadata saved!


In [2]:
import pandas as pd

meta = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv")
isbn_list = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/isbn_list.csv")["ISBN"].tolist()

# Drop duplicate ISBNs but keep the first edition
meta = meta.drop_duplicates(subset="ISBN", keep="first")

# Reorder metadata to match embedding order
meta_aligned = meta.set_index("ISBN").reindex(isbn_list).reset_index()

# Remove rows that still have missing metadata
meta_aligned = meta_aligned.dropna(subset=["Title"])

meta_aligned.to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean_aligned.csv", index=False)

print("✅ Metadata Aligned Successfully!")
print(meta_aligned.head())


✅ Metadata Aligned Successfully!
         ISBN                                              Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

                 Author    Year                Publisher           Genre  \
0    Mark P. O. Morford  2002.0  Oxford University Press  Social Science   
1  Richard Bruce Wright  2001.0    HarperFlamingo Canada       Actresses   
2          Carlo D'Este  1991.0          HarperPerennial         History   
3      Gina Bari Kolata  1999.0     Farrar Straus Giroux         Medical   
4       E. J. W. Barber  1999.0   W. W. Norton & Company          Design   

                                         Description  \
0  Provides an introduction to classical myth

In [6]:
import pandas as pd

df = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final_for_embedding.csv")

df.shape

(271379, 2)

In [9]:
dfx = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean_aligned.csv")

dfx.shape

(251204, 8)

In [ ]:
# dfz = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv",encoding= 'latin-1')

# dfz['text_for_embedding'] = dfz['Title'] + ' ' + dfz['Author'] + ' ' + dfz['Publisher'] + ' ' + dfz['Genre'] + ' ' + dfz['Description']
# dfz[['ISBN', 'text_for_embedding']].to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final_for_embedding.csv", index=False, encoding='latin-1')

In [41]:
import pandas as pd

dfz = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv", encoding='latin-1')
# dfz["text_for_embedding"].duplicated().sum()

# dfz['Genre'] = dfz['Genre'].fillna("Unknown Genre")
# dfz['Description'] = dfz['Description'].fillna("No description available.")
# dfz["Publisher"] = dfz["Publisher"].fillna("Unknown Publisher")
# dfz["Author"] = dfz["Author"].fillna("Unknown Author")

# dfz.drop(columns=["text_for_embedding"], inplace=True)

# dfz.to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv", index=False, encoding='latin-1')

# dfz = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv",encoding= 'latin-1')

dfz['text_for_embedding'] = dfz['Title'] + ' ' + dfz['Author'] + ' ' + dfz['Publisher'] + ' ' + dfz['Genre'] + ' ' + dfz['Description']
dfz[['ISBN', 'Title','Author','Year','Publisher','Genre','Description','text_for_embedding']].to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv", index=False, encoding='latin-1')


In [1]:
import numpy as np
import pandas as pd
import ast

df = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/books_embeddings.csv")

# # Convert string → list → numpy array safely
# embeddings_array = np.array([np.array(ast.literal_eval(e)) for e in df["embeddings"]])

# # Save embeddings
# np.save("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/embeddings.npy", embeddings_array)

# # Save ISBN list
# df["ISBN"].to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/embeddings/isbn_list.csv", index=False)

df.head(5)

,ISBN,embeddings
0,0195153448,"[0.03147442638874054, 0.055889617651700974, 0...."
1,0002005018,"[-0.0006257170462049544, -0.05521390587091446,..."
2,0060973129,"[-0.024097662419080734, 0.058458056300878525, ..."
3,0374157065,"[-0.004065552726387978, 0.013171970844268799, ..."
4,0393045218,"[-0.09412217885255814, 0.13335657119750977, -0..."


In [11]:
import pandas as pd
db = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_with_genre.csv")

db['Publisher'].isna().sum()

0

In [ ]:
db["Genre"].value_counts()

Genre
Historical Fiction                          118342
Children's Books                             41005
Poetry                                       14366
Thriller                                     13778
Romance                                       8819
                                             ...  
Fraud                                            1
Television scripts                               1
Female friendship                                1
Daily telegraph (London, England : 1969)         1
HTML (Document markup language)                  1
Name: count, Length: 1871, dtype: int64

In [13]:
db[["ISBN","text_for_embedding"]].to_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final_for_embedding.csv", index=False)

In [14]:
gm = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_final_for_embedding.csv")

In [ ]:
gm.isnull().sum()

ISBN                  0
text_for_embedding    0
dtype: int64

In [1]:
import csv

input_file = "/home/ayushz/Projects/Books_recommendation_END_TO_END/data/raw/BX-Book-Ratings.csv"
output_file = "/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/cleaned.csv"

with open(input_file, newline='', encoding="utf-8", errors="ignore") as fin, \
     open(output_file, "w", newline='', encoding="utf-8") as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout, quoting=csv.QUOTE_NONE, escapechar='\\')

    # Count expected columns from first row
    first_row = next(reader)
    expected_cols = len(first_row)
    writer.writerow(first_row)

    for row in reader:
        if len(row) == expected_cols:
            writer.writerow(row)


In [5]:
import pandas as pd

rate_x = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/cleaned.csv",encoding = 'latin_1')

rate_x.duplicated().sum()

0

In [1]:
input_file = "/home/ayushz/Projects/Books_recommendation_END_TO_END/data/user/ratings.csv"
output_file = "/home/ayushz/Projects/Books_recommendation_END_TO_END/data/user/cleaned_ratings.csv"

with open(input_file, "r", encoding="utf-8", errors="ignore") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.replace('\\"', '"')   # remove escaped quotes
        line = line.replace(';', ',')     # convert delimiter
        fout.write(line)


In [1]:
import pandas as pd

x = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_final_clean.csv")
y = pd.read_csv("/home/ayushz/Projects/Books_recommendation_END_TO_END/data/processed/books_metadata_with_genre.csv")




Genre
Unknown Genre                      237662
Fiction                              4675
Juvenile Fiction                      852
Biography & Autobiography             627
Juvenile Nonfiction                   302
                                    ...  
Runners (Sports)                        1
Construction workers                    1
Fraud                                   1
Television scripts                      1
HTML (Document markup language)         1
Name: count, Length: 1861, dtype: int64